# Main scan: neighbour-based QA

Loads the LSR-corrected, pair-filtered, cell-combined output from
`main_scan_load.ipynb` (via `artifacts/scan_load_state.pkl`) and runs
neighbour-based QA on the dimensionless ratio R(v_LSR). No temperature
calibration is applied; W_R is in km/s and the neighbour scale floor is
tuned for R-space residuals.

Outputs:
- `artifacts/main_reobserve.json` -- list of cells to reobserve
- `artifacts/spectra_per_session.pdf` -- per-session R(v_LSR) grids,
  with QA-flagged + insufficient-pairs cells excluded

In [ ]:
from utils import (
    compute_cell_metrics,
    neighbor_qa,
    collect_reobserve,
)
from plotters import spectra_per_session_pdf

from pathlib import Path
import json
import pickle
import numpy as np

# --- Hardware ---
HPBW_DEG = 3.4

# --- Per-cell metrics (R-space, no T_B) ---
METRIC_MIN_VALID_CH = 8
METRIC_NOISE_V_MAX_KMS = -100.0
METRIC_SIGNAL_V_LO_KMS = -80.0
METRIC_SIGNAL_V_HI_KMS = 60.0
METRIC_SMOOTH_KERNEL = 5
METRIC_PEAK_MIN_SEP_KMS = 4.0
METRIC_PEAK_PROM_NSIGMA = 2.5
METRIC_MIN_NOISE_CH = 5

# --- Neighbor QA (operates on R; W is in km/s) ---
NEIGHBOR_MAX_SEP_DEG = 2.1
MIN_NEIGHBORS = 2
W_Z_THRESH = 3.0
W_FRAC_THRESH = 0.30
W_SCALE_FLOOR = 5.0
PEAK_V_Z_THRESH = 3.0
PEAK_V_ABS_THRESH = 15.0
PEAK_V_MIN_SIGMA = 3.0
PEAK_V_SCALE_FLOOR = 20.0
BIMODAL_MIN_RATIO = 0.68

# --- Pair-filter spectrum key (must match main_scan_load.ipynb) ---
PAIR_SPECTRUM_KEY = 'R_lsr'

# --- Paths ---
STATE_PATH = Path('artifacts/scan_load_state.pkl')
REOBSERVE_PATH = Path('artifacts/main_reobserve.json')
SPECTRA_PDF_PATH = Path('artifacts/spectra_per_session.pdf')

get_ipython().run_line_magic('matplotlib', 'inline')

## 1. Load state from `main_scan_load.ipynb`

In [ ]:
with open(STATE_PATH, 'rb') as f:
    state = pickle.load(f)

cell_combined = state['cell_combined']
viable_pairs_per_cell = state['viable_pairs_per_cell']
cells_insufficient_pairs = state['cells_insufficient_pairs']
v_lsr_overlap = state['v_lsr_overlap']
dv_kms = state['dv_kms']
sessions = state['sessions']

print(f'Loaded {STATE_PATH} ({STATE_PATH.stat().st_size/1e6:.2f} MB)')
print(f'  {len(cell_combined)} science cells, '
      f'{len(cells_insufficient_pairs)} insufficient-pair cells')
print(f'  dv = {dv_kms:.3f} km/s, v_LSR span '
      f'[{v_lsr_overlap[-1]:.0f}, {v_lsr_overlap[0]:.0f}] km/s, '
      f'{len(sessions)} sessions')

## 2. Neighbour-based QA on integrated W_R and peak velocity

QA operates on the dimensionless ratio R(v_LSR). W is in km/s and the
neighbour scale floor `W_SCALE_FLOOR` is set accordingly (~5 km/s).
Both quantities are sensitive to per-cell / per-session gain and T_sys
differences (W_R amplitude scales with the I_LO2 baseline; peak_v can
swap between near-equal lobes when small bandpass changes tip the
balance). The thresholds below are tuned for R-space; revisit them
once a T_B-calibrated pipeline is in place.

In [ ]:
cell_metrics = compute_cell_metrics(
    cell_combined, v_lsr_overlap, dv_kms,
    min_valid_ch=METRIC_MIN_VALID_CH,
    noise_v_max_kms=METRIC_NOISE_V_MAX_KMS,
    signal_v_lo_kms=METRIC_SIGNAL_V_LO_KMS,
    signal_v_hi_kms=METRIC_SIGNAL_V_HI_KMS,
    smooth_kernel=METRIC_SMOOTH_KERNEL,
    peak_min_sep_kms=METRIC_PEAK_MIN_SEP_KMS,
    peak_prom_nsigma=METRIC_PEAK_PROM_NSIGMA,
    min_noise_ch=METRIC_MIN_NOISE_CH,
)
neighbor_cells = neighbor_qa(
    cell_metrics,
    dv_kms=dv_kms,
    hpbw_deg=HPBW_DEG,
    neighbor_max_sep_deg=NEIGHBOR_MAX_SEP_DEG,
    min_neighbors=MIN_NEIGHBORS,
    w_z_thresh=W_Z_THRESH,
    w_frac_thresh=W_FRAC_THRESH,
    w_scale_floor=W_SCALE_FLOOR,
    peak_v_z_thresh=PEAK_V_Z_THRESH,
    peak_v_abs_thresh=PEAK_V_ABS_THRESH,
    peak_v_min_sigma=PEAK_V_MIN_SIGMA,
    peak_v_scale_floor=PEAK_V_SCALE_FLOOR,
    bimodal_min_ratio=BIMODAL_MIN_RATIO,
)

neighbor_flagged = [c for c in neighbor_cells if c['W_flag'] or c['peak_v_flag']]
print(f'Neighbor QA: {len(neighbor_cells)} cells analyzed (on R)')
print(f'  Flags: W={sum(1 for c in neighbor_cells if c["W_flag"])}, '
      f'peak_v={sum(1 for c in neighbor_cells if c["peak_v_flag"])}, '
      f'any={len(neighbor_flagged)}')

if neighbor_flagged:
    print('  Most deviant cells:')

    def _severity(c):
        w = abs(c['W_frac_resid']) if np.isfinite(c['W_frac_resid']) else 0.0
        v = abs(c['peak_v_z']) if np.isfinite(c['peak_v_z']) else 0.0
        return max(w, v)

    for cell in sorted(neighbor_flagged, key=_severity, reverse=True)[:12]:
        print(
            f"    l={cell['gl']:6.2f} b={cell['gb']:3d} "
            f"W_R={cell['W']:+.2f} km/s (frac={cell['W_frac_resid']:+.2f}, z={cell['W_z']:+.2f}) "
            f"v_peak={cell['peak_v']:+.1f} km/s "
            f"(dv={cell['peak_v_resid']:+.1f}, z={cell['peak_v_z']:+.2f}) "
            f"n={cell['neighbor_count']}"
        )

## 3. Reobserve list

In [ ]:
reobs = collect_reobserve(neighbor_cells, cells_insufficient_pairs)
REOBSERVE_PATH.write_text(json.dumps(reobs, indent=2) + '\n')
print(f'Wrote {REOBSERVE_PATH} -- {len(reobs)} cells')
for r in reobs:
    print(f"  l={r['l']:7.2f} b={r['b']:+3d}  ({r['reason']})")

## 4. Per-session spectra PDF

Renders each session's cells as R(v_LSR) grids, with QA-flagged and
insufficient-pair cells excluded.

In [ ]:
qa_flagged_set = {(c['gl'], c['gb']) for c in neighbor_cells
                  if c['W_flag'] or c['peak_v_flag']}
insufficient_set = {(c['l'], c['b']) for c in cells_insufficient_pairs}
excluded_cells = qa_flagged_set | insufficient_set

n_pages = spectra_per_session_pdf(
    SPECTRA_PDF_PATH,
    v_lsr_overlap,
    viable_pairs_per_cell,
    excluded_cells,
    sessions,
    spectrum_key=PAIR_SPECTRUM_KEY,
)
print(f'Saved {SPECTRA_PDF_PATH} ({n_pages} pages); '
      f'excluded {len(excluded_cells)} cells '
      f'({len(qa_flagged_set)} QA + {len(insufficient_set)} insufficient-pairs)')